# Language Modeling with a Vanilla RNN

**Question:** Demonstrate the concept of language modeling using RNN, which involves training a model to predict the next word in a sequence of words.

---

## What is Language Modeling?
A **language model** learns the statistical patterns of a language so it can predict:
> *Given the words so far, what word comes next?*

Example: `"the movie was"` → model predicts → `"fantastic"` or `"boring"`

This is the foundation of autocomplete, ChatGPT, and every modern NLP system.

---

## What is an RNN (Recurrent Neural Network)?
A regular neural network processes one fixed-size input. An RNN processes **sequences** — it reads one word at a time and carries a **hidden state** (its memory) from one word to the next.

```
word1 --> [RNN cell] --> hidden state h1
word2 --> [RNN cell] --> hidden state h2   (uses h1 as memory)
word3 --> [RNN cell] --> hidden state h3   (uses h2 as memory)
                              |
                         final output
```

After reading all words in the sequence, the final hidden state `h3` summarises everything the RNN has seen — we use that to predict the next word.

---

## What is an Embedding?
Words are strings — neural networks need numbers. An **embedding** converts each word index into a small vector of floats (like a fingerprint for that word).

- `"movie"` → index 7 → `[0.23, -0.11, 0.84, ...]`  (a 32-number vector)

These vectors are **learned during training** — similar words end up with similar vectors.

In [ ]:
import torch
import torch.nn as nn

## Step 1 - Build the Vocabulary

We start with a tiny corpus (6 sentences about movies). From this we build two lookup tables:
- **`w2i`** (word-to-index): converts a word string to a unique integer
- **`i2w`** (index-to-word): the reverse — converts an integer back to a word

Index `0` is reserved for **padding** (a dummy token used to make all sequences the same length).
Real words start from index `1`.

The `|` operator merges two dictionaries (Python 3.9+): `{a:1} | {b:2}` gives `{a:1, b:2}`.

In [ ]:
# Our tiny training dataset - 6 movie review sentences
corpus = [
    'the movie was fantastic',
    'the movie was boring',
    'i loved the acting',
    'i hated the plot',
    'the plot was amazing',
    'the acting was terrible'
]

# Get every unique word across all sentences, sorted alphabetically
words = sorted(set(' '.join(corpus).split()))
print('Vocabulary:', words)

# Build word->index map: words get indices 1,2,3,...
# The | merges in {'': 0} so that index 0 = padding token
w2i = {w: i+1 for i, w in enumerate(words)} | {'': 0}

# Build reverse map: index->word (used to decode predictions)
i2w = {i: w for w, i in w2i.items()}

vocab_size = len(w2i)  # Total number of tokens including padding
max_len    = max(len(line.split()) for line in corpus)  # Length of the longest sentence

print(f'Vocab size: {vocab_size}, Max sentence length: {max_len}')
print('Word-to-index:', w2i)

## Step 2 - Build Training Pairs (N-grams with Padding)

For each sentence, we create multiple training examples by taking every prefix and asking:
> *Given these words, predict the next one.*

Example for `"the movie was fantastic"` (indices: `[5, 7, 9, 4]`):

| Input X (context)       | Target y (next word) |
|-------------------------|---------------------|
| `[0, 0, 0, 5]`          | `7` (movie)         |
| `[0, 0, 5, 7]`          | `9` (was)           |
| `[0, 5, 7, 9]`          | `4` (fantastic)     |

**Why left-padding with zeros?**
The RNN needs all inputs to be the same length. We pad with `0`s on the **left** (before the words)
so the real words always appear at the end — this way the final hidden state reflects the most recent context.

In [ ]:
X, y = [], []

for line in corpus:
    # Convert each word in the sentence to its index
    tokens = [w2i[w] for w in line.split()]

    # For each position i, create a (context -> next word) training pair
    for i in range(1, len(tokens)):
        # Take the first i+1 tokens, then left-pad with zeros to reach max_len
        prefix = tokens[:i+1]
        padded = [0] * (max_len - len(prefix)) + prefix  # [0,0,..., word1, word2, ...]

        X.append(padded[:-1])   # Everything except the last word = input context
        y.append(padded[-1])    # The last word = the word we want to predict

X_train = torch.tensor(X)  # Shape: (num_examples, max_len - 1)
y_train = torch.tensor(y)  # Shape: (num_examples,)

print(f'Training examples: {len(X_train)}')
print(f'Input shape: {X_train.shape}  (examples x sequence_length)')
print(f'First example  X={X_train[0].tolist()}  ->  y={y_train[0].item()} ({i2w[y_train[0].item()]})')

## Step 3 - Define the RNN Model

The model has three layers:

1. **`nn.Embedding(vocab_size, 32)`**
   Converts word indices to 32-dimensional vectors. Input: `(batch, seq_len)` of integers. Output: `(batch, seq_len, 32)` of floats.

2. **`nn.RNN(32, 32, batch_first=True)`**
   The recurrent layer. Reads the embedded sequence one step at a time, updating its hidden state.
   - Input size = 32 (embedding dimension)
   - Hidden size = 32
   - `batch_first=True` means input shape is `(batch, seq_len, features)` — more intuitive
   - Returns two things: `output` (all hidden states) and `h` (just the final hidden state). We only need `h`.

3. **`nn.Linear(32, vocab_size)`**
   Maps the final hidden state (32 numbers) to a score for every word in the vocabulary.
   The word with the highest score is the prediction.

```
Input indices  [0, 5, 7, 9]
     |  Embedding
     v
Vectors  [[...], [0.23,-0.1,...], [0.5,0.2,...], [0.1,-0.3,...]]
     |  RNN (reads left to right, updates hidden state each step)
     v
Final hidden state  [0.1, -0.4, 0.7, ...]  (32 numbers summarising the sequence)
     |  Linear
     v
Scores for each word  [1.2, 0.3, -0.5, 2.1, ...]  (one per vocab word)
     -> argmax -> predicted word index
```

In [ ]:
class TextRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Layer 1: convert word indices to 32-d vectors
        self.embed = nn.Embedding(vocab_size, 32)

        # Layer 2: vanilla RNN — processes the sequence and produces a hidden state
        # batch_first=True means input is (batch_size, seq_len, embed_dim)
        self.rnn = nn.RNN(input_size=32, hidden_size=32, batch_first=True)

        # Layer 3: map the final hidden state to a score for every vocabulary word
        self.fc = nn.Linear(32, vocab_size)

    def forward(self, x):
        embedded = self.embed(x)           # (batch, seq_len) -> (batch, seq_len, 32)
        _, h = self.rnn(embedded)          # h shape: (num_layers, batch, 32) — we discard output
        final_hidden = h[-1]               # Take the last layer's hidden state: (batch, 32)
        scores = self.fc(final_hidden)     # (batch, 32) -> (batch, vocab_size)
        return scores

## Step 4 - Train the Model

We train for 300 epochs. Because our dataset is tiny (18 examples), we feed **all examples at once** each epoch (no batching needed).

**CrossEntropyLoss** is used because this is a classification problem — we're classifying each context into one of `vocab_size` possible next words.

You should see the loss steadily decrease, meaning the model is getting better at predicting the next word.

In [ ]:
model     = TextRNN(vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()  # Combines softmax + log loss; good for multi-class classification

for epoch in range(300):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)           # Forward pass: get word scores for every training example
    loss    = criterion(outputs, y_train)  # Compare predicted scores to true next words

    loss.backward()                   # Backpropagate through time (BPTT)
    optimizer.step()                  # Update weights

    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1}/300 | Loss: {loss.item():.4f}')

## Step 5 - Predict the Next Word

Given a seed phrase (e.g. `"the movie was"`), the model predicts what word comes next.

Steps inside `predict()`:
1. Split the seed into words and convert each to its index using `w2i`
2. Left-pad so the sequence length matches what the model was trained on
3. Run through the model → get scores for every vocabulary word
4. Pick the word with the highest score (`argmax`)
5. Convert that index back to a word using `i2w`

`w2i.get(w, 0)` means: look up word `w` in the dictionary; if not found, return `0` (the padding token). This handles unknown words gracefully.

In [ ]:
def predict(seed_phrase):
    # Convert each seed word to its index (0 if the word is unknown)
    tokens = [w2i.get(w, 0) for w in seed_phrase.lower().split()]

    # Left-pad so the input length = max_len - 1 (same as training)
    padded = [0] * (max_len - 1 - len(tokens)) + tokens

    with torch.no_grad():  # No gradient needed for prediction
        scores     = model(torch.tensor([padded]))  # Add batch dimension with []
        pred_index = scores.argmax(dim=1).item()    # Index of highest-scoring word

    return i2w.get(pred_index, '<unknown>')

# Test the model
print('the movie was  ->', predict('the movie was'))
print('i loved the    ->', predict('i loved the'))
print('the plot was   ->', predict('the plot was'))
print('i hated the    ->', predict('i hated the'))